# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Finding #1 — Growing vs. declining content: The paper reports that growing pages were younger on average than declining pages (185 days versus 228 days), while their average word counts were similar. I’d like to understand how “growing” and “declining” were defined from the traffic data. Could you also clarify whether the analysis accounts for multiple pages belonging to the same client?

Finding #4 — Freshness and growth: The paper reports a 5.43:1 growth-to-decline ratio for pages updated 31–90 days earlier, along with a separate comparison of older refreshed pages and a comparison group. How were growth, decline, and the comparison groups defined? Could other differences between those pages help explain the observed impression gap?

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [1]:
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit, train_test_split

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
df.head()

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [2]:
df["trend_outcome"] = (df["trend_direction"] == "down").astype(int)

In [3]:
random_train_idx, random_test_idx = train_test_split(
    df.index,
    test_size=0.20,
    random_state=42,
    stratify=df["trend_outcome"],
)

In [4]:
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42,
)
train_indices, test_indices = next(
    splitter.split(
        df,
        y=df["trend_outcome"],
        groups=df["client_id"],
    )
)

In [5]:
group_train_df = df.iloc[train_indices]
group_test_df = df.iloc[test_indices]
train_clients = set(group_train_df["client_id"])
test_clients = set(group_test_df["client_id"])
client_overlap = train_clients.intersection(test_clients)

print("Grouped split client overlap:", len(client_overlap))
print("Random train rows:", len(random_train_idx))
print("Random test rows:", len(random_test_idx))
print("Random test decline rate:",
      df.iloc[random_test_idx]["trend_outcome"].mean())
print("Grouped train rows:", len(group_train_df))
print("Grouped test rows:", len(group_test_df))
print("Grouped test decline rate:",
      group_test_df["trend_outcome"].mean())

Grouped split client overlap: 0
Random train rows: 24000
Random test rows: 6000
Random test decline rate: 0.542
Grouped train rows: 23837
Grouped test rows: 6163
Grouped test decline rate: 0.5109524582184002


In [6]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Exclude direct label fields, IDs, and the 30-day trend inputs.
# The 90-day aggregates remain here so their overlapping-window risk
# can be audited explicitly below.
prohibited_columns = {
    "trend_outcome", "trend_direction", "trend_pct",
    "client_id", "content_id", "provider_used", "model_used",
    "impressions_last_30d", "impressions_prev_30d",
    "clicks_last_30d", "clicks_prev_30d",
    "sessions_last_30d", "sessions_prev_30d",
}
feature_columns = [
    col for col in df.columns if col not in prohibited_columns
]

prediction_outputs = {}

def score_split(train_idx, test_idx, split_name):
    train_df = df.iloc[train_idx]
    test_df = df.iloc[test_idx]

    x_train = train_df[feature_columns]
    x_test = test_df[feature_columns]
    y_train = train_df["trend_outcome"]
    y_test = test_df["trend_outcome"]

    numeric_cols = x_train.select_dtypes(include=["number", "bool"]).columns
    categorical_cols = x_train.select_dtypes(include=["object"]).columns

    preprocessing = ColumnTransformer([
        ("numeric", Pipeline([
            ("impute", SimpleImputer(strategy="median")),
            ("scale", StandardScaler()),
        ]), numeric_cols),
        ("categorical", Pipeline([
            ("impute", SimpleImputer(strategy="constant", fill_value="missing")),
            ("encode", OneHotEncoder(handle_unknown="ignore")),
        ]), categorical_cols),
    ])

    model = Pipeline([
        ("preprocess", preprocessing),
        ("classifier", LogisticRegression(max_iter=1000, random_state=42)),
    ])

    model.fit(x_train, y_train)
    probabilities = model.predict_proba(x_test)[:, 1]

    # Save a small, safe set of held-out predictions for error review.
    prediction_frame = test_df[[
        "content_id", "trend_outcome", "impressions_90d", "ctr", "avg_position"
    ]].copy()
    prediction_frame["predicted_probability"] = probabilities
    prediction_frame["predicted_outcome"] = (probabilities >= 0.5).astype(int)
    prediction_frame["error_type"] = "correct"
    prediction_frame.loc[
        (prediction_frame["trend_outcome"] == 0)
        & (prediction_frame["predicted_outcome"] == 1),
        "error_type",
    ] = "false_positive"
    prediction_frame.loc[
        (prediction_frame["trend_outcome"] == 1)
        & (prediction_frame["predicted_outcome"] == 0),
        "error_type",
    ] = "false_negative"
    prediction_outputs[split_name] = prediction_frame

    return {
        "split": split_name,
        "test_rows": len(test_df),
        "test_decline_rate": y_test.mean(),
        "average_precision": average_precision_score(y_test, probabilities),
    }

results = pd.DataFrame([
    score_split(random_train_idx, random_test_idx, "Random row split"),
    score_split(train_indices, test_indices, "Client-grouped split"),
])

results

,split,test_rows,test_decline_rate,average_precision
0,Random row split,6000,0.542000,0.708260
1,Client-grouped split,6163,0.510952,0.575788


In [7]:
# Review a few errors on clients held out from training. The 0.5 cutoff
# is used here only to label false positives and false negatives.
grouped_predictions = prediction_outputs["Client-grouped split"]
false_positives = (
    grouped_predictions[grouped_predictions["error_type"] == "false_positive"]
    .sort_values("predicted_probability", ascending=False)
    .head(3)
)
false_negatives = (
    grouped_predictions[grouped_predictions["error_type"] == "false_negative"]
    .sort_values("predicted_probability", ascending=True)
    .head(3)
)

error_examples = pd.concat([false_positives, false_negatives], ignore_index=True)
error_examples[[
    "content_id", "error_type", "trend_outcome",
    "predicted_outcome", "predicted_probability",
    "impressions_90d", "ctr", "avg_position",
]]

,content_id,error_type,trend_outcome,predicted_outcome,predicted_probability,impressions_90d,ctr,avg_position
0,content_374e795aab68,false_positive,0,1,0.919979,235,0.85,31.0
1,content_7be5f150dc65,false_positive,0,1,0.906494,290,0.00,5.9
2,content_41baf0722ad9,false_positive,0,1,0.901719,3115,0.00,12.8
3,content_e18144cbd19d,false_negative,1,0,0.079905,3,0.00,2.0
4,content_917fc1b11fe1,false_negative,1,0,0.081552,916,0.00,78.6
5,content_742a8fcba2fe,false_negative,1,0,0.083448,643,0.00,76.0


Average precision was 0.708 on the random split and 0.576 on the client-grouped split. The corresponding test decline rates were 0.542 and 0.511. The grouped-split score was lower in this comparison, but the decline rates also differed, so the entire gap cannot be attributed to client grouping alone. This measures ranking of the observed decline label; it does not show that the model predicts a future decline.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [8]:
# These fields are measured over a window that may overlap the
# period used to create trend_outcome. Check them against the data dictionary.
overlap_candidates = {
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "ctr",
    "avg_position",
    "days_with_impressions",
    "days_with_sessions",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
    "impression_tier",
    "position_tier",
}

feature_audit = pd.DataFrame({
    "feature": feature_columns,
    "audit_note": [
        (
            "Overlapping-window risk: check whether this includes "
            "the period used to define trend_outcome"
        )
        if (
            col in overlap_candidates
            or col.endswith("_90d")
            or col.endswith("_last_30d")
            or col.endswith("_prev_30d")
        )
        else "Check the data dictionary to confirm this was available at decision time"
        for col in feature_columns
    ],
})

feature_audit

,feature,audit_note
0,search_volume,Check the data dictionary to confirm this was ...
1,competition,Check the data dictionary to confirm this was ...
2,competition_level,Check the data dictionary to confirm this was ...
3,cpc,Check the data dictionary to confirm this was ...
4,content_type,Check the data dictionary to confirm this was ...
5,main_intent,Check the data dictionary to confirm this was ...
6,word_count,Check the data dictionary to confirm this was ...
7,char_count,Check the data dictionary to confirm this was ...
8,impressions_90d,Overlapping-window risk: check whether this in...
9,clicks_90d,Overlapping-window risk: check whether this in...


### Leakage audit

I excluded `trend_direction` and `trend_pct`, which define the decline label, along with client and content IDs and the 30-day trend input fields. I did not find those direct label fields in the model’s feature list.

However, the model uses 90-day measurements and features derived from them: impressions, clicks, sessions, pageviews, users, engagement and scroll rates, AI traffic, CTR, average position, and their tiers. The 90-day window includes the last 30 days and the previous 30 days used to define `trend_outcome`. These features therefore overlap with the label period and may contain information about the observed outcome.

The client-grouped split tests performance on clients not seen during training, but it does not remove this timing overlap. I interpret the scores as measuring classification of the observed decline label using same-window information, not prediction of future decline. Content age, freshness, word count, and keyword metrics should only be treated as decision-time features if they were available at the intended review time.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

### Claim rewrite

**Earlier claim:** The model is useful as a ranking aid for human review.

**Rewritten claim:** In this sample, average precision was 0.708 on a random row split and 0.576 on a client-grouped split, with test decline rates of 0.542 and 0.511, respectively. The grouped-split score was lower in this comparison, though the test decline rates also differed. Because several features overlap with the period used to define the decline label, these results measure classification of an observed trend using same-window information; they do not show that the model predicts future declines or that refreshing a page will improve its performance.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.